# Batch Approach

In [8]:
import requests
from requests.adapters import HTTPAdapter, Retry
import json
import re

The following url was generate using the query:

(taxonomy_id:2759) AND (existence:1) AND (reviewed:true) AND (fragment:false) AND (length:[40 TO *]) AND ((cc_scl_term_exp:SL-0091) OR (cc_scl_term_exp:SL-0191) OR (cc_scl_term_exp:SL-0173) OR (cc_scl_term_exp:SL-0039) OR (cc_scl_term_exp:SL-0209) OR (cc_scl_term_exp:SL-0204)) NOT (ft_signal:*) NOT ((cc_scl_term:SL-0095) OR (cc_scl_term:SL-0132) OR (cc_scl_term:SL-0112) OR (cc_scl_term:SL-0243))

In [9]:
url = "https://rest.uniprot.org/uniprotkb/search?format=json&query=%28%28taxonomy_id%3A2759%29+AND+%28existence%3A1%29+AND+%28reviewed%3Atrue%29+AND+%28fragment%3Afalse%29+AND+%28length%3A%5B40+TO+*%5D%29+AND+%28%28cc_scl_term_exp%3ASL-0091%29+OR+%28cc_scl_term_exp%3ASL-0191%29+OR+%28cc_scl_term_exp%3ASL-0173%29+OR+%28cc_scl_term_exp%3ASL-0039%29+OR+%28cc_scl_term_exp%3ASL-0209%29+OR+%28cc_scl_term_exp%3ASL-0204%29%29+NOT+%28ft_signal%3A*%29+NOT+%28%28cc_scl_term%3ASL-0095%29+OR+%28cc_scl_term%3ASL-0132%29+OR+%28cc_scl_term%3ASL-0112%29+OR+%28cc_scl_term%3ASL-0243%29%29%29"
batch_size = 500

In [10]:
retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

In [11]:
def get_next_link(headers):
    if "Link" in headers:
        # The regular expression is used to extract the next link for pagination
        re_next_link = re.compile(r'<(.+)>; rel="next"')
        match = re_next_link.match(headers["Link"])
        if match:
            return match.group(1)

# This function actually retrieve the next data batch in the search.
# The function act as an iterator, yielding the next result batch at every call
# The function terminates after the last batch has been returned. In this case,
# the next link will be None
def get_batch(batch_url):
    while batch_url:
        # Run the API call
        response = session.get(batch_url)
        # Will raise an error if an error status code is obtained
        response.raise_for_status()
        # Get the total number of entries in the search
        total = response.headers["x-total-results"]
        # Yield the response and the total number of entries
        yield response, total
        # Get the link to the API call for the next data batch
        batch_url = get_next_link(response.headers)

In [ ]:
def extract_fields(entry):

    accession = entry.get("primaryAccession", "")
    organism_name = entry.get("organism", {}).get("scientificName", "")
    length = entry.get("sequence", {}).get("length", 0)
    sequence = entry.get("sequence", {}).get("value", "")

    #kindom
    lineage = entry.get("organism", {}).get("lineage", [])
    if "Metazoa" in lineage:
        kingdom = "Metazoa"
    elif "Fungi" in lineage:
        kingdom = "Fungi"
    elif "Viridiplantae" in lineage:
        kingdom = "Plants"
    else:
        kingdom = "Other"
#controls is a transmembrane helix is present in the first 90 amino acids of the sequence, starts with false,if it is present returns true
    early_trans = False

    for f in entry.get("features", []):
        if f.get("type") == "Transmembrane" and f.get("description") == "Helical":
            try:
                start_val = f["location"]["start"]["value"]
                if start_val < 90:
                    early_trans = True
                    break
            except KeyError:

                continue

    return (accession, organism_name, kingdom, length, early_trans, sequence)


In [ ]:

def get_dataset(search_url, extract_function, output_file_name):
    filtered_json = []
    n_total, n_filtered = 0, 0

    # Run the API call in batches
    for batch, total in get_batch(search_url):
        # parse the JSON body of the response
        batch_json = json.loads(batch.text)

#iterate over the entries in the batch and apply the filter function to each entry
        for entry in batch_json.get("results", []):
            n_total += 1
            filtered_json.append(extract_function(entry))
            n_filtered += 1

    print(n_total, n_filtered)
#generates the output files in TSV and FASTA format
    tsv = f"{output_file_name}.tsv"

    fasta = f"{output_file_name}.fasta"

    with open(tsv, "w") as ofs:
        for entry in filtered_json:
            # Print the fields in TSV format,k considering only the first 5 fields (accession, organism name, kingdom, length, early_trans)
            print(*entry[:5], sep="\t", file=ofs)


    with open(fasta, "w") as ofs:
      for entry in filtered_json:
            # Print the fields in FASTA format, considering only the first and last fields (accession and sequence)
            print(f">{entry[0]}", file=ofs)
            print(entry[-1], file=ofs)
#in this file is NOT present the cleavage column , unified with the positive dataset. 

In [14]:
output_file = "dataset_neg"
get_dataset(url, extract_fields, output_file)

19594 19594
